<a href="https://colab.research.google.com/github/mantrikaran/F1.Stats.Guy/blob/main/Agent1A_Production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# AGENT 1A — MILESTONE DETECTION RUNNER
# Scans all .sql files in the SQL folder, runs each against the
# latest completed race, and appends results to Google Sheets.
# ═══════════════════════════════════════════════════════════════════════

# ── INSTALL + MOUNT ───────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "gspread", "duckdb", "-q"])

from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import duckdb
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
from datetime import date, datetime

# ═══════════════════════════════════════════════════════════════════════
# CONFIG — only edit this block
# ═══════════════════════════════════════════════════════════════════════

DB_FOLDER  = "/content/drive/MyDrive/F1 Stats Guy - Jolpica/Jolpica Database"
SQL_FOLDER = "/content/drive/MyDrive/F1 Stats Guy - Jolpica/Codes/SQL Codes"

SHEET_ID   = "1wlZBishj6NAs4wU_qNaWKfURw03IY_gXMUipxWhWVXg"
SHEET_GID  = 1623554071  # Stat Table tab

# ── MANUAL OVERRIDE (set both to None for auto-detect) ────────────────
# Example: OVERRIDE_SEASON = 2026, OVERRIDE_ROUND = 5
OVERRIDE_SEASON = None
OVERRIDE_ROUND  = None

# ── OUTPUT SCHEMA (must match sheet column order exactly) ─────────────
SHEET_COLUMNS = [
    "Stat ID",
    "Date Generated",
    "Race Year",
    "Stat Description",
    "Driver Name",
    "Constructor Name",
    "Race Name",
    "Race Track",
    "Track Country",
    "Metric",
    "Dimension",
]

# ═══════════════════════════════════════════════════════════════════════
# STEP 1 — CONNECT DUCKDB TO CSV FILES
# ═══════════════════════════════════════════════════════════════════════

print("═" * 60)
print("AGENT 1A — Milestone Detection Runner")
print("═" * 60)

con = duckdb.connect()

# Register all CSVs as views so SQL files can reference them by name
csv_tables = {
    "race_results":               "race_results.csv",
    "rounds":                     "rounds.csv",
    "drivers":                    "drivers.csv",
    "teams":                      "teams.csv",
    "qualifying_results":         "qualifying_results.csv",
    "sprint_results":             "sprint_results.csv",
    "sprint_qualifying_results":  "sprint_qualifying_results.csv",
}

for table_name, filename in csv_tables.items():
    filepath = f"{DB_FOLDER}/{filename}"
    if not os.path.exists(filepath):
        print(f"  ⚠ Missing CSV: {filename} — skipping")
        continue
    con.execute(f"""
        CREATE OR REPLACE VIEW {table_name} AS
        SELECT * FROM read_csv_auto('{filepath}', header=True)
    """)

print(f"  ✅ DuckDB connected — {len(csv_tables)} tables registered\n")

# ═══════════════════════════════════════════════════════════════════════
# STEP 2 — RESOLVE TRIGGER ROUND
# ═══════════════════════════════════════════════════════════════════════

today = date.today().isoformat()

if OVERRIDE_SEASON is not None and OVERRIDE_ROUND is not None:
    # Manual override path
    trigger_df = con.execute(f"""
        SELECT round_id, year AS season, round_number, round_name,
               circuit_name, country_code, race_date
        FROM   rounds
        WHERE  year         = {OVERRIDE_SEASON}
        AND    round_number = {OVERRIDE_ROUND}
        LIMIT  1
    """).df()
    mode = f"MANUAL OVERRIDE — {OVERRIDE_SEASON} Round {OVERRIDE_ROUND}"
else:
    # Auto-detect: latest race where race_date <= today
    trigger_df = con.execute(f"""
        SELECT round_id, year AS season, round_number, round_name,
               circuit_name, country_code, race_date
        FROM   rounds
        WHERE  race_date <= '{today}'
        ORDER  BY year DESC, round_number DESC
        LIMIT  1
    """).df()
    mode = "AUTO-DETECT"

if trigger_df.empty:
    raise ValueError("No completed race found. Set OVERRIDE_SEASON and OVERRIDE_ROUND manually.")

trigger        = trigger_df.iloc[0]
trigger_round_id   = trigger["round_id"]
trigger_season     = int(trigger["season"])
trigger_round_num  = int(trigger["round_number"])
trigger_race_name  = trigger["round_name"]
trigger_circuit    = trigger["circuit_name"]
trigger_country    = trigger["country_code"]
trigger_race_date  = trigger["race_date"]

print(f"  Mode           : {mode}")
print(f"  Trigger Race   : {trigger_race_name} ({trigger_season}, Round {trigger_round_num})")
print(f"  Race Date      : {trigger_race_date}")
print(f"  Circuit        : {trigger_circuit} ({trigger_country})")
print(f"  round_id       : {trigger_round_id}\n")

# ═══════════════════════════════════════════════════════════════════════
# STEP 3 — DISCOVER AND RUN ALL SQL FILES
# ═══════════════════════════════════════════════════════════════════════

sql_files = sorted(glob.glob(f"{SQL_FOLDER}/*.sql"))

if not sql_files:
    raise FileNotFoundError(f"No .sql files found in: {SQL_FOLDER}")

print(f"  Found {len(sql_files)} SQL file(s):\n")
for f in sql_files:
    print(f"    • {os.path.basename(f)}")
print()

all_results = []
run_log     = []

for sql_path in sql_files:
    filename = os.path.basename(sql_path)
    print(f"  Running: {filename}")

    try:
        with open(sql_path, "r") as f:
            sql = f.read()

        # Inject trigger_round_id as a safe string literal
        sql_injected = sql.replace("$trigger_round_id", f"'{trigger_round_id}'")

        result_df = con.execute(sql_injected).df()
        row_count = len(result_df)
        print(f"    → {row_count} milestone(s) detected")

        if row_count > 0:
            result_df["_source_file"] = filename
            all_results.append(result_df)

        run_log.append({"file": filename, "status": "OK", "rows": row_count})

    except Exception as e:
        print(f"    ⚠ ERROR: {e}")
        run_log.append({"file": filename, "status": f"ERROR: {e}", "rows": 0})

print()

# ═══════════════════════════════════════════════════════════════════════
# STEP 4 — COMBINE AND MAP TO SHEET SCHEMA
# ═══════════════════════════════════════════════════════════════════════

def derive_metric(milestone_id):
    """Derive Metric column from milestone_id prefix."""
    if isinstance(milestone_id, str):
        prefix = milestone_id[0].upper()
        if prefix == "M":
            return "Win"
        elif prefix == "P":
            return "Podium"
        elif prefix == "T":
            return "H2H"
    return "Other"

if not all_results:
    print("  ⚠ No milestones detected across all SQL files. Nothing to write.")
else:
    combined_df = pd.concat(all_results, ignore_index=True)

    print(f"  Total milestones across all files: {len(combined_df)}\n")

    # Map to sheet schema
    today_str = datetime.now().strftime("%Y-%m-%d")

    output_df = pd.DataFrame({
        "Stat ID":          combined_df["milestone_id"],
        "Date Generated":   today_str,
        "Race Year":        combined_df["season"].astype(int),
        "Stat Description": combined_df["milestone_label"],
        "Driver Name":      combined_df["driver_name"],
        "Constructor Name": combined_df["team_name"],
        "Race Name":        combined_df["trigger_race_name"],
        "Race Track":       trigger_circuit,   # from rounds.csv via trigger resolution
        "Track Country":    trigger_country,   # from rounds.csv via trigger resolution
        "Metric":           combined_df["milestone_id"].apply(derive_metric),
        "Dimension":        "Post Race",
    })

    # Enforce column order
    output_df = output_df[SHEET_COLUMNS]

    print("  Sample output (first 3 rows):")
    print(output_df.head(3).to_string(index=False))
    print()

# ═══════════════════════════════════════════════════════════════════════
# STEP 5 — APPEND TO GOOGLE SHEETS
# ═══════════════════════════════════════════════════════════════════════

    print("  Authenticating with Google Sheets...")
    auth.authenticate_user()
    creds, _ = default()
    gc       = gspread.authorize(creds)

    sh       = gc.open_by_key(SHEET_ID)

    # Find the correct worksheet by GID
    target_ws = None
    for ws in sh.worksheets():
        if ws.id == SHEET_GID:
            target_ws = ws
            break

    if target_ws is None:
        raise ValueError(f"Worksheet with GID {SHEET_GID} not found in spreadsheet.")

    # Append rows (values only, no header)
    rows_to_append = output_df.astype(str).values.tolist()
    target_ws.append_rows(rows_to_append, value_input_option="USER_ENTERED")

    print(f"  ✅ {len(rows_to_append)} row(s) appended to Google Sheets\n")

# ═══════════════════════════════════════════════════════════════════════
# STEP 6 — RUN SUMMARY
# ═══════════════════════════════════════════════════════════════════════

print("═" * 60)
print("  RUN SUMMARY")
print("═" * 60)
print(f"  Race          : {trigger_race_name} ({trigger_season} R{trigger_round_num})")
print(f"  SQL files run : {len(sql_files)}")
print()
for entry in run_log:
    status_icon = "✅" if entry["status"] == "OK" else "⚠"
    print(f"  {status_icon}  {entry['file']:<55} {entry['rows']} row(s)")
print()
total_written = sum(e["rows"] for e in run_log if e["status"] == "OK")
print(f"  Total milestones written to sheet: {total_written}")
print("═" * 60)